In [1]:
from io import StringIO
import time
import os
import requests
import pandas as pd
import numpy as np

In [2]:
# URL of the page with the list of IPOS
url = "https://www.iposcoop.com/ipos-recently-filed/"

# Setting the header for the HTTP request
headers = {'User-Agent': 'Mozilla/5.0'}

In [3]:
# Making the HTTP request to get the page content
response = requests.get(url, headers=headers)

# Reading the tables from the HTML page using pandas
tables = pd.read_html(StringIO(response.text))

In [4]:
# Displaying the number of tables found
print(len(tables))   

# Displaying the first rows of the first table
tables[0].head()

1


,File Date,Company,Symbol,Managers,Shares (millions),Price Low,Price High,Est $ Vol (millions),Expected To Trade,SCOOP Rating
0,2026-09-24,ADARx Pharmaceuticals,ADRX,J.P.Morgan/Morgan Stanley/TD Cowen/UBS Investm...,26.25,$17.00,$17.00,$446.25,Priced,S/O
1,2026-09-24,Bluerock Acquisition Corp. II,BRRKU,BTIG,15.00,$10.00,$10.00,$150.00,Priced,S/O
2,2026-09-24,City Therapeutics,CTY,Goldman Sachs/Jefferies/Stifel/Oppenheimer & Co.,0.00,NaN,NaN,$100.00,TBA,S/O
3,2026-09-24,Hacker Interstellar,SOUD,Kingswood Capital Markets,3.90,$10.50,$11.50,$42.90,TBA,S/O
4,2026-09-24,New Iceland Arctic Acquisition Corp.,NIAAU,Chardan,12.50,$10.00,$10.00,$125.00,Week of,S/O


In [5]:
tables

[      File Date                               Company Symbol  \
 0    2026-09-24                 ADARx Pharmaceuticals   ADRX   
 1    2026-09-24         Bluerock Acquisition Corp. II  BRRKU   
 2    2026-09-24                     City Therapeutics    CTY   
 3    2026-09-24                   Hacker Interstellar   SOUD   
 4    2026-09-24  New Iceland Arctic Acquisition Corp.  NIAAU   
 ..          ...                                   ...    ...   
 495  2026-01-05                            Buda Juice   BUDA   
 496  2025-12-30                            Optimal AI    TBA   
 497  2025-12-29                  Cafe Deco Group Ltd.    TBA   
 498  2025-12-29                       Riverstone Ltd.   DNJF   
 499  2025-12-23                             AOJE Inc.   AOJE   
 
                                               Managers  Shares (millions)  \
 0    J.P.Morgan/Morgan Stanley/TD Cowen/UBS Investm...              26.25   
 1                                                 BTIG       

In [6]:
df = tables[0]

In [7]:
df.to_csv(r'../data/ipo.csv', index=False)

In [8]:
# Function to get the names of all tables in a given URL
def scrapping(urls, table):
    # URL of the page with the list of companies
    url = urls

    # Setting the header for the HTTP request
    headers = {'User-Agent': 'Mozilla/5.0'}
    
    # Making the HTTP request to get the page content
    response = requests.get(url, headers=headers)
    
    # Checking whether the table is present in the page content
    check_response = table in response.text
    
    # Reading the tables from the HTML page using pandas
    if check_response == True:
        print(f'Table {table} status: {check_response}')
        # Reading the tables from the HTML page using pandas
        tables = pd.read_html(StringIO(response.text), match=table)
        df = tables[0]  
    
    return df


In [9]:
# Function to get the names of the tables on an HTML page
def get_html_tables_name(urls):
    """
    Function to get the names of the tables on an HTML page.
    
    Parameters:
    - response: HTTP response object containing the page content.
    
    Returns:
    - List with the names of the tables found on the page.
    """
    
    # URL of the page with the list of companies
    url = urls

    # Setting the header for the HTTP request
    headers = {'User-Agent': 'Mozilla/5.0'}

    # Making the HTTP request to get the page content
    response = requests.get(url, headers=headers)
    
    # Reading the tables from the HTML page using pandas
    tables = pd.read_html(StringIO(response.text))
    print(len(tables))

    # Displaying the number of tables found and the first 5 columns of each table
    for i, t in enumerate(tables):
        print(i, t.shape, list(t.columns)[:5])


In [10]:
check_tables_names = get_html_tables_name('https://www.iposcoop.com/ipos-recently-filed/')

1
0 (500, 10) ['File Date', 'Company', 'Symbol', 'Managers', 'Shares (millions)']


In [11]:
check_tables = scrapping(0,'https://www.iposcoop.com/ipos-recently-filed/')

MissingSchema: Invalid URL '0': No scheme supplied. Perhaps you meant https://0?

---

### Question 1

In [12]:
df = pd.read_csv(r'../data/ipo.csv')

In [13]:
df

,File Date,Company,Symbol,Managers,Shares (millions),Price Low,Price High,Est $ Vol (millions),Expected To Trade,SCOOP Rating
0,2026-09-24,ADARx Pharmaceuticals,ADRX,J.P.Morgan/Morgan Stanley/TD Cowen/UBS Investm...,26.25,$17.00,$17.00,$446.25,Priced,S/O
1,2026-09-24,Bluerock Acquisition Corp. II,BRRKU,BTIG,15.00,$10.00,$10.00,$150.00,Priced,S/O
2,2026-09-24,City Therapeutics,CTY,Goldman Sachs/Jefferies/Stifel/Oppenheimer & Co.,0.00,NaN,NaN,$100.00,TBA,S/O
3,2026-09-24,Hacker Interstellar,SOUD,Kingswood Capital Markets,3.90,$10.50,$11.50,$42.90,TBA,S/O
4,2026-09-24,New Iceland Arctic Acquisition Corp.,NIAAU,Chardan,12.50,$10.00,$10.00,$125.00,Week of,S/O
...,...,...,...,...,...,...,...,...,...,...
495,2026-01-05,Buda Juice,BUDA,MDB Capital Group,2.70,$7.50,$7.50,$20.03,Priced,S/O
496,2025-12-30,Optimal AI,TBA,R.F. Lafferty & Co.,2.50,$4.00,$5.00,$11.25,TBA,S/O
497,2025-12-29,Cafe Deco Group Ltd.,TBA,Kingswood Capital Markets,1.80,$6.00,$8.00,$12.60,TBA,S/O
498,2025-12-29,Riverstone Ltd.,DNJF,Joseph Gunnar,2.50,$5.00,$7.00,$15.00,TBA,S/O


In [14]:
file_date_parsed = pd.to_datetime(df['File Date'])
df_witdrawn = df[(df['Expected To Trade'] == 'Withdrawn') & (file_date_parsed < '2026-09-11')]

In [15]:
df_witdrawn.reset_index(inplace=True)

In [16]:
df_witdrawn

,index,File Date,Company,Symbol,Managers,Shares (millions),Price Low,Price High,Est $ Vol (millions),Expected To Trade,SCOOP Rating
0,38,2026-09-10,Motive Technologies (Withdrawn),MTVE,J.P.Morgan/Citigroup/Barclays/Jefferies/RBC Ca...,0.00,NaN,NaN,$100.00,Withdrawn,S/O
1,43,2026-09-04,Idea Tech Holding (Withdrawn),IDTL,R.F. Lafferty & Co.,2.00,$4.00,$5.00,$9.00,Withdrawn,S/O
2,50,2026-09-01,Hornbeck Offshore Services (Withdrawn),HOS,J.P. Morgan/ Barclays/DNB Markets/Piper Sandle...,0.00,NaN,NaN,$100.00,Withdrawn,S/O
3,52,2026-08-31,Coolbit Technologies Ltd. (Withdrawn),CBAI,Eddid Securities USA,5.00,$4.00,$5.00,$22.50,Withdrawn,S/O
4,59,2026-08-27,Timber Road Acquisition Corp. (Withdrawn),TMRDU,Roth Capital Partners/Stone X Financial Inc.,20.00,$10.00,$10.00,$200.00,Withdrawn,S/O
5,69,2026-08-24,pan-Africa Corp. (Withdrawn),BLEUU,Santander,12.50,$10.00,$10.00,$125.00,Withdrawn,S/O
6,84,2026-08-18,Living Homeopathy (Withdrawn),LHI,Revere Securities,3.75,$4.00,$6.00,$19.00,Withdrawn,S/O
7,128,2026-07-31,Nuclea Energy (Withdrawn),NCLA,Joseph Gunnar & Co.,5.60,$8.00,$10.00,$50.40,Withdrawn,S/O
8,158,2026-07-10,APEX Global Solutions Ltd. (WITHDRAWN),APEX,US Tiger Securities,3.75,$4.00,$4.00,$15.00,Withdrawn,S/O
9,163,2026-07-08,Cloud Data Holdings (Withdrawn),CDN,Cathay Securities,3.75,$4.00,$4.50,$15.94,Withdrawn,S/O


In [17]:
conditions = [
    df_witdrawn['Company'].str.contains('Technologies', case=False, na=False),
    df_witdrawn['Company'].str.contains(r'Acquisition Corp|Acquisition Corporation|\bCorp\b|Corp\.', case=False, na=False),
    df_witdrawn['Company'].str.contains('Inc|Incorporated', case=False, na=False),
    df_witdrawn['Company'].str.contains('Group', case=False, na=False),
    df_witdrawn['Company'].str.contains('Ltd|Limited', case=False, na=False),
    df_witdrawn['Company'].str.contains('Holdings|Holding', case=False, na=False)
]

In [18]:
choices = [
    'Technologies',
    'Acquisition Corp',
    'Inc.',
    'Group',
    'Limited',
    'Holdings'
]

In [19]:
df_witdrawn['Company Type'] = np.select(conditions, choices, default='Other')

In [20]:
df_witdrawn.head(3)

,index,File Date,Company,Symbol,Managers,Shares (millions),Price Low,Price High,Est $ Vol (millions),Expected To Trade,SCOOP Rating,Company Type
0,38,2026-09-10,Motive Technologies (Withdrawn),MTVE,J.P.Morgan/Citigroup/Barclays/Jefferies/RBC Ca...,0.0,NaN,NaN,$100.00,Withdrawn,S/O,Technologies
1,43,2026-09-04,Idea Tech Holding (Withdrawn),IDTL,R.F. Lafferty & Co.,2.0,$4.00,$5.00,$9.00,Withdrawn,S/O,Holdings
2,50,2026-09-01,Hornbeck Offshore Services (Withdrawn),HOS,J.P. Morgan/ Barclays/DNB Markets/Piper Sandle...,0.0,NaN,NaN,$100.00,Withdrawn,S/O,Other


In [21]:
df_witdrawn.info()

<class 'pandas.DataFrame'>
RangeIndex: 31 entries, 0 to 30
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   index                 31 non-null     int64  
 1   File Date             31 non-null     str    
 2   Company               31 non-null     str    
 3   Symbol                31 non-null     str    
 4   Managers              31 non-null     str    
 5   Shares (millions)     31 non-null     float64
 6   Price Low             25 non-null     str    
 7   Price High            25 non-null     str    
 8   Est $ Vol (millions)  31 non-null     str    
 9   Expected To Trade     31 non-null     str    
 10  SCOOP Rating          31 non-null     str    
 11  Company Type          31 non-null     str    
dtypes: float64(1), int64(1), str(10)
memory usage: 3.0 KB


In [22]:
for col in ['Price High', 'Price Low', 'Est $ Vol (millions)', 'Shares (millions)']:
    df_witdrawn[col] = (
        df_witdrawn[col]
        .astype(str)
        .str.replace('$', '', regex=False)
        .str.replace(',', '', regex=False)
        .str.strip()
    )
    df_witdrawn[col] = pd.to_numeric(df_witdrawn[col], errors='coerce')

In [23]:
df_witdrawn

,index,File Date,Company,Symbol,Managers,Shares (millions),Price Low,Price High,Est $ Vol (millions),Expected To Trade,SCOOP Rating,Company Type
0,38,2026-09-10,Motive Technologies (Withdrawn),MTVE,J.P.Morgan/Citigroup/Barclays/Jefferies/RBC Ca...,0.00,NaN,NaN,100.00,Withdrawn,S/O,Technologies
1,43,2026-09-04,Idea Tech Holding (Withdrawn),IDTL,R.F. Lafferty & Co.,2.00,4.0,5.0,9.00,Withdrawn,S/O,Holdings
2,50,2026-09-01,Hornbeck Offshore Services (Withdrawn),HOS,J.P. Morgan/ Barclays/DNB Markets/Piper Sandle...,0.00,NaN,NaN,100.00,Withdrawn,S/O,Other
3,52,2026-08-31,Coolbit Technologies Ltd. (Withdrawn),CBAI,Eddid Securities USA,5.00,4.0,5.0,22.50,Withdrawn,S/O,Technologies
4,59,2026-08-27,Timber Road Acquisition Corp. (Withdrawn),TMRDU,Roth Capital Partners/Stone X Financial Inc.,20.00,10.0,10.0,200.00,Withdrawn,S/O,Acquisition Corp
5,69,2026-08-24,pan-Africa Corp. (Withdrawn),BLEUU,Santander,12.50,10.0,10.0,125.00,Withdrawn,S/O,Acquisition Corp
6,84,2026-08-18,Living Homeopathy (Withdrawn),LHI,Revere Securities,3.75,4.0,6.0,19.00,Withdrawn,S/O,Other
7,128,2026-07-31,Nuclea Energy (Withdrawn),NCLA,Joseph Gunnar & Co.,5.60,8.0,10.0,50.40,Withdrawn,S/O,Other
8,158,2026-07-10,APEX Global Solutions Ltd. (WITHDRAWN),APEX,US Tiger Securities,3.75,4.0,4.0,15.00,Withdrawn,S/O,Limited
9,163,2026-07-08,Cloud Data Holdings (Withdrawn),CDN,Cathay Securities,3.75,4.0,4.5,15.94,Withdrawn,S/O,Holdings


In [24]:
df_witdrawn['Avg Price'] = (df_witdrawn['Price High'] + df_witdrawn['Price Low']) / 2

In [25]:
df_witdrawn

,index,File Date,Company,Symbol,Managers,Shares (millions),Price Low,Price High,Est $ Vol (millions),Expected To Trade,SCOOP Rating,Company Type,Avg Price
0,38,2026-09-10,Motive Technologies (Withdrawn),MTVE,J.P.Morgan/Citigroup/Barclays/Jefferies/RBC Ca...,0.00,NaN,NaN,100.00,Withdrawn,S/O,Technologies,NaN
1,43,2026-09-04,Idea Tech Holding (Withdrawn),IDTL,R.F. Lafferty & Co.,2.00,4.0,5.0,9.00,Withdrawn,S/O,Holdings,4.50
2,50,2026-09-01,Hornbeck Offshore Services (Withdrawn),HOS,J.P. Morgan/ Barclays/DNB Markets/Piper Sandle...,0.00,NaN,NaN,100.00,Withdrawn,S/O,Other,NaN
3,52,2026-08-31,Coolbit Technologies Ltd. (Withdrawn),CBAI,Eddid Securities USA,5.00,4.0,5.0,22.50,Withdrawn,S/O,Technologies,4.50
4,59,2026-08-27,Timber Road Acquisition Corp. (Withdrawn),TMRDU,Roth Capital Partners/Stone X Financial Inc.,20.00,10.0,10.0,200.00,Withdrawn,S/O,Acquisition Corp,10.00
5,69,2026-08-24,pan-Africa Corp. (Withdrawn),BLEUU,Santander,12.50,10.0,10.0,125.00,Withdrawn,S/O,Acquisition Corp,10.00
6,84,2026-08-18,Living Homeopathy (Withdrawn),LHI,Revere Securities,3.75,4.0,6.0,19.00,Withdrawn,S/O,Other,5.00
7,128,2026-07-31,Nuclea Energy (Withdrawn),NCLA,Joseph Gunnar & Co.,5.60,8.0,10.0,50.40,Withdrawn,S/O,Other,9.00
8,158,2026-07-10,APEX Global Solutions Ltd. (WITHDRAWN),APEX,US Tiger Securities,3.75,4.0,4.0,15.00,Withdrawn,S/O,Limited,4.00
9,163,2026-07-08,Cloud Data Holdings (Withdrawn),CDN,Cathay Securities,3.75,4.0,4.5,15.94,Withdrawn,S/O,Holdings,4.25


In [26]:
calculated_value = df_witdrawn['Shares (millions)'] * df_witdrawn['Avg Price']

In [27]:
df_witdrawn['Shares_offered_value'] = np.where(
    calculated_value.notna(),
    calculated_value,
    df_witdrawn['Est $ Vol (millions)']
)

In [28]:
df_witdrawn.head(3)

,index,File Date,Company,Symbol,Managers,Shares (millions),Price Low,Price High,Est $ Vol (millions),Expected To Trade,SCOOP Rating,Company Type,Avg Price,Shares_offered_value
0,38,2026-09-10,Motive Technologies (Withdrawn),MTVE,J.P.Morgan/Citigroup/Barclays/Jefferies/RBC Ca...,0.0,NaN,NaN,100.0,Withdrawn,S/O,Technologies,NaN,100.0
1,43,2026-09-04,Idea Tech Holding (Withdrawn),IDTL,R.F. Lafferty & Co.,2.0,4.0,5.0,9.0,Withdrawn,S/O,Holdings,4.5,9.0
2,50,2026-09-01,Hornbeck Offshore Services (Withdrawn),HOS,J.P. Morgan/ Barclays/DNB Markets/Piper Sandle...,0.0,NaN,NaN,100.0,Withdrawn,S/O,Other,NaN,100.0


In [29]:
df_grouped = (
    df_witdrawn
    .groupby('Company Type')['Shares_offered_value']
    .sum()
    .reset_index()
)

In [30]:
df_grouped = df_grouped.sort_values(by='Shares_offered_value', ascending=False)

In [31]:
df_grouped

,Company Type,Shares_offered_value
0,Acquisition Corp,499.9850
3,Inc.,351.0000
2,Holdings,311.6575
5,Other,290.4450
4,Limited,197.2500
6,Technologies,184.9000
1,Group,32.5000


---

### Question 2

In [32]:
# Core imports for scraping, timing, and data analysis
from io import StringIO
import time
import os
import requests
import pandas as pd
import numpy as np
import yfinance as yf
from scipy.stats import pearsonr

In [33]:
url_q2 = 'https://www.iposcoop.com/2025-pricings/'

headers = {'User-Agent': 'Mozilla/5.0'}

# Fazendo a requisição HTTP para obter o conteúdo da página
response = requests.get(url_q2, headers=headers)

# Lendo TODAS as tabelas <table> da página HTML.
# pd.read_html retorna uma LISTA de DataFrames (uma entrada por tabela encontrada).
tables = pd.read_html(StringIO(response.text))

# Quantas tabelas vieram
print(len(tables))

# Espiar cada tabela para descobrir qual é a que você quer
for i, t in enumerate(tables):
    print(i, t.shape, list(t.columns)[:5])

# Selecionar a 1ª tabela da lista como o DataFrame de trabalho
df_ipo = tables[0]

1
0 (231, 10) ['Company', 'Symbol', 'Industry', 'Offer Date', 'Shares (millions)']


In [34]:
df_ipo.head(3)

,Company,Symbol,Industry,Offer Date,Shares (millions),Offer Price,1st Day Close,Current Price,Return,SCOOP Rating
0,Vine Hill Capital Investment Corp. II,VHCPU,Blank Check,12/18/2025,20.0,$10.00,$0.00,$0.00,0.00%,S/O
1,Andersen Group,ANDG,Consumer Services,12/17/2025,11.0,$16.00,$23.50,$23.79,48.69%,S/O
2,Medline Inc.,MDLN,Health Care,12/17/2025,216.0,$29.00,$41.00,$43.83,51.14%,S/O


In [35]:
df_ipo.info()

<class 'pandas.DataFrame'>
RangeIndex: 231 entries, 0 to 230
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Company            231 non-null    str    
 1   Symbol             231 non-null    str    
 2   Industry           231 non-null    str    
 3   Offer Date         231 non-null    str    
 4   Shares (millions)  231 non-null    float64
 5   Offer Price        231 non-null    str    
 6   1st Day Close      231 non-null    str    
 7   Current Price      231 non-null    str    
 8   Return             231 non-null    str    
 9   SCOOP Rating       231 non-null    str    
dtypes: float64(1), str(9)
memory usage: 18.2 KB


In [36]:
df_ipo['Offer Date'] = pd.to_datetime(df_ipo['Offer Date'], format='%m/%d/%Y')
ipo = df_ipo[df_ipo['Offer Date'] < '2025-09-01']
ipo = ipo.reset_index(drop=True)

In [37]:
ipo['Return'] = ipo['Return'].str.rstrip('%').astype(float)
ipo = ipo[ipo['Return'] != 0]

In [38]:
ipo.head(3)

,Company,Symbol,Industry,Offer Date,Shares (millions),Offer Price,1st Day Close,Current Price,Return,SCOOP Rating
0,"Picard Medical, Inc.",PMI,Health Care,2025-08-29,4.3,$4.00,$4.59,$1.68,-58.00,S/O
1,GrowHub Limited (The),TGHL,Technology,2025-08-28,3.7,$4.00,$3.38,$0.40,-90.00,S/O
2,TryHard Holdings Limited,THH,Consumer Services,2025-08-28,1.5,$4.00,$6.02,$29.47,636.75,S/O


In [39]:
tickers_list = ipo['Symbol'].tolist()

result = {}
failed_tickers = []

for ticker in tickers_list:
    data = yf.download(ticker, start='2000-01-01', end='2026-09-12', multi_level_index=False)
    if data.empty:
        failed_tickers.append(ticker)
    else:
        result[ticker] = data

stocks_df = pd.concat(result, names=['Ticker', 'Date']).reset_index()

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********

In [40]:
stocks_df['Growth_252d'] = stocks_df['Close'] / stocks_df.groupby('Ticker')['Close'].shift(252)

In [41]:
stocks_df['Volatility'] = stocks_df.groupby('Ticker')['Close'].rolling(30).std().reset_index(level=0, drop=True) * np.sqrt(252)


In [42]:
stocks_df['Sharpe'] = (stocks_df['Growth_252d'] - 0.05) / stocks_df['Volatility']

In [43]:
final_day = stocks_df[stocks_df['Date'] == '2026-09-11']
print('Número de tickers com dado em 2026-09-11:', len(final_day))

display(final_day[['Growth_252d', 'Volatility', 'Sharpe']].describe())

median_sharpe = final_day['Sharpe'].median()
print(f'Mediana do Sharpe em 2026-09-11: {median_sharpe:.4f}')

Número de tickers com dado em 2026-09-11: 132


c:\Users\leona\Documents\stock-markets-analytics-zoomcamp-2026\.venv\Lib\site-packages\pandas\core\nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


,Growth_252d,Volatility,Sharpe
count,130.000000,131.000000,130.000000
mean,1.057613,27.516340,inf
std,3.049007,66.879183,NaN
min,0.001005,0.000000,-0.040147
25%,0.141035,2.301309,0.012042
50%,0.594534,7.885748,0.050142
75%,1.038821,23.452319,0.133619
max,33.638270,627.698106,inf


Mediana do Sharpe em 2026-09-11: 0.0501


---

### Question 3

In [44]:
for m in range(1, 13):
    days = m * 21
    stocks_df[f'future_growth_{m}_m'] = stocks_df.groupby('Ticker')['Close'].shift(-days) / stocks_df['Close']

In [45]:
stocks_df.head(3)

,Ticker,Date,Close,High,Low,Open,Volume,Growth_252d,Volatility,Sharpe,...,future_growth_3_m,future_growth_4_m,future_growth_5_m,future_growth_6_m,future_growth_7_m,future_growth_8_m,future_growth_9_m,future_growth_10_m,future_growth_11_m,future_growth_12_m
0,PMI,2025-08-29,229.5,356.0,209.5,210.5,126794,NaN,NaN,NaN,...,0.464052,0.381264,0.381264,0.209150,0.228758,0.095643,0.058824,0.036819,0.022353,0.016819
1,PMI,2025-09-02,257.0,267.5,232.5,242.5,98358,NaN,NaN,NaN,...,0.412451,0.340467,0.402724,0.221790,0.185019,0.068482,0.059922,0.034047,0.016732,0.017821
2,PMI,2025-09-03,264.5,270.0,250.5,257.0,35960,NaN,NaN,NaN,...,0.374291,0.344045,0.362949,0.202268,0.181474,0.039698,0.060491,0.031758,0.013724,0.016975


In [46]:
min_date_df = stocks_df.loc[
    stocks_df.groupby('Ticker')['Date'].idxmin(),
    ['Ticker', 'Date', 'Close']
].rename(columns={'Date': 'min_date', 'Close': 'min_date_close'})

In [47]:
min_date_df

,Ticker,min_date,min_date_close
59601,AAPG,2025-01-24,17.379999
53986,AARD,2025-02-13,14.310000
43787,ADVB,2025-03-06,73.000000
30494,AII,2025-05-08,16.008226
25970,AIRO,2025-06-13,24.000000
...,...,...,...
13841,WYFI,2025-08-07,16.219999
55174,XHLD,2025-02-13,102.000000
15518,YMAT,2025-07-30,20.400000
1578,YMT,2025-08-19,672.000000


In [48]:
entry_growth_df = stocks_df.merge(
    min_date_df,
    left_on=['Ticker', 'Date'],
    right_on=['Ticker', 'min_date'],
    how='inner'
)

In [49]:
future_growth_cols = [f'future_growth_{m}_m' for m in range(1, 13)]
stats = entry_growth_df[future_growth_cols].describe()
print(stats)

median_row = stats.loc['50%']
best_month = median_row.idxmax()
print(f'Melhor mês: {best_month} — mediana: {median_row.max():.4f}')

       future_growth_1_m  future_growth_2_m  future_growth_3_m  \
count         132.000000         131.000000         131.000000   
mean            1.125337           1.310796           1.263011   
std             1.253840           1.961863           1.913413   
min             0.072941           0.106197           0.090147   
25%             0.710433           0.584863           0.442477   
50%             0.940843           0.892958           0.828743   
75%             1.103369           1.164977           1.172574   
max             9.466019          16.066783          16.344464   

       future_growth_4_m  future_growth_5_m  future_growth_6_m  \
count         131.000000         131.000000         131.000000   
mean            1.144200           1.093135           1.124265   
std             1.558346           1.469040           1.668720   
min             0.063927           0.032245           0.022075   
25%             0.392759           0.400704           0.308026   
50%      